## A charged particle in a magnetic field

The vector Lorentz force is
$$\mathbf{F}=q(\mathbf{E}+\mathbf{v}\times\mathbf{B}),\qquad
\frac{d\mathbf{x}}{dt}=\mathbf{v},\qquad
\frac{d\mathbf{v}}{dt}=\frac{\mathbf{F}}{m}.$$

We use normalized units and uniform, constant fields. With $\mathbf{E}=0$, the magnetic force changes the direction of velocity without changing its magnitude, producing a helix. The Boris update preserves speed in this case. Edit `E`, `B`, or `v_0` below to explore other trajectories.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.ticker import MaxNLocator
from IPython.display import HTML, Video, display


In [ ]:
# Normalized units; change these to explore the motion.
q, m = 1.0, 1.0
E = np.array([0.0, 0.0, 0.1])
B = np.array([0.0, 0.0, 1.0])
v_0 = np.array([2.0, 2.0, 0.0])
x_0 = np.array([1.0, 0.0, 0.0])


In [ ]:
def lorentz_force(q, v, E, B):
    return q * (np.asarray(E) + np.cross(v, B))


def lorentz_motion(q, m, v_0, E, B, *, x_0=(0, 0, 0), dt=0.01, t_max=12):
    """Boris velocity update and trapezoidal position update; uniform fields."""
    if m <= 0 or dt <= 0 or t_max <= 0:
        raise ValueError("m, dt and t_max must be positive.")
    E, B = np.asarray(E, dtype=float), np.asarray(B, dtype=float)
    t = np.linspace(0, t_max, int(np.ceil(t_max / dt)) + 1)
    x = np.zeros((len(t), 3))
    v = np.zeros_like(x)
    x[0], v[0] = x_0, v_0

    for i, step in enumerate(np.diff(t)):
        v_minus = v[i] + q * E * step / (2 * m)
        rotation = q * B * step / (2 * m)
        s = 2 * rotation / (1 + np.dot(rotation, rotation))
        v_prime = v_minus + np.cross(v_minus, rotation)
        v_plus = v_minus + np.cross(v_prime, s)
        v[i + 1] = v_plus + q * E * step / (2 * m)
        x[i + 1] = x[i] + step * (v[i] + v[i + 1]) / 2

    return x, v, t



x, v, t = lorentz_motion(q, m, v_0, E, B, x_0=x_0, dt=0.01, t_max=12)

In [ ]:
# Interactive 3D movie with play/pause and a draggable time slider.
with plt.rc_context({"font.family": "DejaVu Sans", "font.size": 10,
                     "animation.embed_limit": 80, "grid.color": "#28405a",
                     "grid.alpha": 0.35, "grid.linewidth": 0.5}):
    fig = plt.figure(figsize=(8.5, 6.5), dpi=90, facecolor="#080e1c")
    ax = fig.add_subplot(111, projection="3d", facecolor="#080e1c")
    fig.subplots_adjust(left=0.02, right=0.98, bottom=0.07, top=0.84)
    fig.text(0.08, 0.94, "LORENTZ / IN MOTION", fontsize=10,
             color="#6ce6ef", weight="bold")
    fig.text(0.08, 0.885, "The geometry of a magnetic force", fontsize=21,
             color="#edf5ff", weight="bold")
    fig.text(0.08, 0.035, "BORIS INTEGRATION   /   NORMALIZED UNITS",
             fontsize=9, color="#879bb5")

    # Fit the view to the trajectory, also when field parameters are changed.
    spans = np.ptp(x, axis=0)
    padding = max(float(spans.max()) * 0.12, 0.2)
    lower, upper = x.min(axis=0) - padding, x.max(axis=0) + padding
    ax.set(xlim=(lower[0], upper[0]), ylim=(lower[1], upper[1]),
           zlim=(lower[2], upper[2]))
    ax.set_box_aspect(upper - lower)
    ax.set_proj_type("ortho")
    ax.view_init(elev=22, azim=-55)
    for axis, name in zip((ax.xaxis, ax.yaxis, ax.zaxis), ("x", "y", "z")):
        axis.set_pane_color((0.05, 0.09, 0.16, 1))
        axis.label.set_color("#b7c9df")
        axis.set_label_text(name)
        axis.labelpad = 8
        axis.set_major_locator(MaxNLocator(4))
        axis.line.set_color("#28405a")
    ax.tick_params(colors="#879bb5", labelsize=8)
    ax.grid(alpha=0.15)

    # Arrows show the direction of B at a small grid of locations.
    field_norm = np.linalg.norm(B)
    if field_norm > 0:
        gx, gy = np.meshgrid(np.linspace(lower[0], upper[0], 3),
                             np.linspace(lower[1], upper[1], 3))
        gz = np.full_like(gx, lower[2])
        direction = B / field_norm
        ax.quiver(gx, gy, gz, *[np.full_like(gx, b) for b in direction],
                  length=0.8 * float(spans.max() + padding),
                  color="#5bbdcc", alpha=0.18, linewidth=0.8,
                  arrow_length_ratio=0.08)
    ax.plot(*x.T, color="#6ce6ef", alpha=0.12, lw=1)
    trail_glow, = ax.plot([], [], [], color="#6ce6ef", lw=8, alpha=0.07)
    trail, = ax.plot([], [], [], color="#6ce6ef", lw=2.2, alpha=0.9)
    halo, = ax.plot([], [], [], "o", ms=25, color="#ff85c8", alpha=0.09)
    ball, = ax.plot([], [], [], "o", ms=9, color="#ffd0e9",
                    markeredgecolor="#ff85c8", markeredgewidth=1.5)
    velocity_line, = ax.plot([], [], [], color="#ffc56e", lw=2)
    readout = ax.text2D(0.02, 0.92, "", transform=ax.transAxes,
                        color="#d3e5f8", family="monospace", va="top")
    ax.text2D(0.02, 0.08, "CYAN  trajectory / B\nGOLD  velocity",
              transform=ax.transAxes, color="#879bb5", fontsize=9)
    fps = 24
    frames = np.unique(np.r_[np.searchsorted(t, np.arange(0, t[-1], 1 / fps)),
                             len(t) - 1])

    def update(i):
        start = np.searchsorted(t, max(0, t[i] - 4))
        for line in (trail_glow, trail):
            line.set_data_3d(*x[start:i + 1].T)
        for point in (halo, ball):
            point.set_data_3d(*x[i:i + 1].T)
        velocity_line.set_data_3d(*np.vstack([x[i], x[i] + 0.3 * v[i]]).T)
        readout.set_text(f"t      {t[i]:5.2f}\n|v|    {np.linalg.norm(v[i]):5.3f}\n"
                         f"q / m  {q / m:+5.2f}")
        return trail_glow, trail, halo, ball, velocity_line, readout

    animation = FuncAnimation(fig, update, frames=frames, interval=1000 / fps,
                               blit=False, repeat=True)
    player = animation.to_jshtml(default_mode="loop")
    plt.close(fig)

display(HTML(player))


In [ ]:
# Save the interactive HTML player and MP4 video.
import imageio_ffmpeg
from pathlib import Path
from lorentz_force_player import make_lorentz_player

controlled_player = make_lorentz_player(x, v, t, E, B, q, m)
Path("lorentz_force.html").write_text(controlled_player, encoding="utf-8")

with plt.rc_context({"animation.ffmpeg_path": imageio_ffmpeg.get_ffmpeg_exe()}):
    animation.save("lorentz_force.mp4", writer=FFMpegWriter(fps=fps), dpi=150)

display(Video("lorentz_force.mp4", embed=True))
